## 3. Build Silver Cleaned Admissions Table

Create business-readable fields for mental health status, employment status, wait-time tier, operational risk profile, and treatment intensity.

In [7]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_tedsa_admissions_2023")

silver_df = (
    bronze_df
    .withColumn(
        "PSYPROB_Label",
        F.when(F.col("PSYPROB") == 1, "Co-occurring mental health problem")
         .when(F.col("PSYPROB") == 2, "No co-occurring mental health problem")
         .otherwise("Unknown")
    )
    .withColumn(
        "EMPLOY_Label",
        F.when(F.col("EMPLOY") == 1, "Full-time employment")
         .when(F.col("EMPLOY") == 2, "Part-time employment")
         .when(F.col("EMPLOY") == 3, "Unemployed")
         .when(F.col("EMPLOY") == 4, "Not in labor force")
         .otherwise("Unknown")
    )
    .withColumn(
        "Wait_Time_Tier",
        F.when(F.col("DAYWAIT") == 0, "Same day")
         .when(F.col("DAYWAIT") == 1, "Short wait (1-7 days)")
         .when(F.col("DAYWAIT") == 2, "Moderate wait (8-14 days)")
         .when(F.col("DAYWAIT") == 3, "Long wait (15-30 days)")
         .when(F.col("DAYWAIT") == 4, "Extended wait (31+ days)")
         .otherwise("Unknown")
    )
    .withColumn(
        "Risk_Profile",
        F.when(
            (F.col("PSYPROB") == 1) & (F.col("EMPLOY").isin(3, 4)),
            "High Risk"
        )
        .when(
            ((F.col("PSYPROB") == 1) & (F.col("EMPLOY") == 2)) |
            ((F.col("PSYPROB") == 2) & (F.col("EMPLOY") == 3)),
            "Medium Risk"
        )
        .when(
            F.col("EMPLOY") == 1,
            "Low Risk"
        )
        .otherwise("Unknown")
    )
    .withColumn(
        "Treatment_Intensity",
        F.when(F.col("SERVICES") == 7, "Low Intensity")
         .when(F.col("SERVICES").isin(1, 2, 6), "Medium Intensity")
         .when(F.col("SERVICES").isin(3, 4, 5, 8), "High Intensity")
         .otherwise("Unknown")
    )
)

StatementMeta(, a94ada23-d6fc-470a-95cb-d80c1adf2e17, 9, Finished, Available, Finished, False)

In [8]:
silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_tedsa_admissions_cleaned")

StatementMeta(, a94ada23-d6fc-470a-95cb-d80c1adf2e17, 10, Finished, Available, Finished, False)

In [ ]:
silver_check = spark.table("silver_tedsa_admissions_cleaned")

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

## 4. Build Gold KPI Summary Table

Calculate the core operational KPIs used in the dashboard, including high-risk rate, treatment mismatch rate, alignment rate, and delayed admission rate.

In [10]:
from pyspark.sql import functions as F

silver_df = spark.table("silver_tedsa_admissions_cleaned")

total_admissions = silver_df.select("CASEID").distinct().count()

high_risk_admissions = (
    silver_df
    .filter(F.col("Risk_Profile") == "High Risk")
    .select("CASEID")
    .distinct()
    .count()
)

high_risk_low_intensity_admissions = (
    silver_df
    .filter(
        (F.col("Risk_Profile") == "High Risk") &
        (F.col("Treatment_Intensity") == "Low Intensity")
    )
    .select("CASEID")
    .distinct()
    .count()
)

aligned_high_risk_admissions = (
    silver_df
    .filter(
        (F.col("Risk_Profile") == "High Risk") &
        (F.col("Treatment_Intensity") != "Low Intensity")
    )
    .select("CASEID")
    .distinct()
    .count()
)

high_risk_delayed_admissions = (
    silver_df
    .filter(
        (F.col("Risk_Profile") == "High Risk") &
        (F.col("DAYWAIT").isin(2, 3, 4))
    )
    .select("CASEID")
    .distinct()
    .count()
)

gold_kpi_summary = spark.createDataFrame(
    [
        ("Total Admissions", total_admissions, None),
        ("High-Risk Admissions", high_risk_admissions, None),
        ("High Risk Rate", high_risk_admissions, high_risk_admissions / total_admissions),
        ("High-Risk Low-Intensity Admissions", high_risk_low_intensity_admissions, None),
        ("Treatment Mismatch Rate", high_risk_low_intensity_admissions, high_risk_low_intensity_admissions / high_risk_admissions),
        ("High-Risk Alignment Rate", aligned_high_risk_admissions, aligned_high_risk_admissions / high_risk_admissions),
        ("High-Risk Delayed Admission Rate", high_risk_delayed_admissions, high_risk_delayed_admissions / high_risk_admissions),
    ],
    ["Metric", "Value", "Rate"]
)

gold_kpi_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_kpi_summary")

StatementMeta(, a94ada23-d6fc-470a-95cb-d80c1adf2e17, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd7fffb5-0761-40ee-afb6-8b485b8fe482)

## 5. Build Gold Dashboard-Ready Aggregate Tables

Create aggregate tables that support dashboard visuals for risk-treatment alignment, wait-time monitoring, and treatment distribution.

In [11]:
from pyspark.sql import functions as F

silver_df = spark.table("silver_tedsa_admissions_cleaned")

gold_risk_treatment_matrix = (
    silver_df
    .groupBy("Risk_Profile", "Treatment_Intensity")
    .agg(F.countDistinct("CASEID").alias("Admissions"))
    .orderBy("Risk_Profile", "Treatment_Intensity")
)

gold_wait_time_by_risk = (
    silver_df
    .filter(F.col("Wait_Time_Tier") != "Unknown")
    .groupBy("Risk_Profile", "Wait_Time_Tier", "DAYWAIT")
    .agg(F.countDistinct("CASEID").alias("Admissions"))
    .orderBy("Risk_Profile", "DAYWAIT")
)

gold_treatment_distribution = (
    silver_df
    .groupBy("SERVICES", "Treatment_Intensity")
    .agg(F.countDistinct("CASEID").alias("Admissions"))
    .orderBy("SERVICES")
)

gold_risk_treatment_matrix.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_risk_treatment_matrix")

gold_wait_time_by_risk.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_wait_time_by_risk")

gold_treatment_distribution.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_treatment_distribution")

StatementMeta(, a94ada23-d6fc-470a-95cb-d80c1adf2e17, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d35c8bbe-9f2b-4931-afef-05d153e7f034)

SynapseWidget(Synapse.DataFrame, f06a18a1-229e-49f0-b97c-d3eb79f97238)

SynapseWidget(Synapse.DataFrame, 110e943c-293b-4692-bdb3-16e3f4a802c5)

## 6. Build Employment Access Friction Gold Table

Create an employment-by-treatment-intensity table to monitor whether patients with co-occurring mental health needs are routed into lower-intensity care differently by employment status.

In [1]:
from pyspark.sql import functions as F

silver_df = spark.table("silver_tedsa_admissions_cleaned")

cooccurring_df = (
    silver_df
    .filter(F.col("PSYPROB") == 1)
    .filter(F.col("EMPLOY_Label") != "Unknown")
    .filter(F.col("Treatment_Intensity").isNotNull())
)

employment_totals = (
    cooccurring_df
    .groupBy("EMPLOY_Label")
    .agg(F.countDistinct("CASEID").alias("Employment_Total_Admissions"))
)

low_intensity_by_employment = (
    cooccurring_df
    .filter(F.col("Treatment_Intensity") == "Low Intensity")
    .groupBy("EMPLOY_Label")
    .agg(F.countDistinct("CASEID").alias("Low_Intensity_Admissions"))
)

gold_employment_treatment_mix = (
    cooccurring_df
    .groupBy("EMPLOY_Label", "Treatment_Intensity")
    .agg(F.countDistinct("CASEID").alias("Admissions"))
    .join(employment_totals, on="EMPLOY_Label", how="left")
    .join(low_intensity_by_employment, on="EMPLOY_Label", how="left")
    .withColumn(
        "Mismatch_Rate",
        F.col("Low_Intensity_Admissions") / F.col("Employment_Total_Admissions")
    )
    .withColumn(
        "Employment_Sort_Order",
        F.when(F.col("EMPLOY_Label") == "Part-time employment", 1)
         .when(F.col("EMPLOY_Label") == "Full-time employment", 2)
         .when(F.col("EMPLOY_Label") == "Not in labor force", 3)
         .when(F.col("EMPLOY_Label") == "Unemployed", 4)
         .otherwise(99)
    )
    .withColumn(
        "Treatment_Intensity_Sort_Order",
        F.when(F.col("Treatment_Intensity") == "Low Intensity", 1)
         .when(F.col("Treatment_Intensity") == "Medium Intensity", 2)
         .when(F.col("Treatment_Intensity") == "High Intensity", 3)
         .otherwise(99)
    )
)

(
    gold_employment_treatment_mix.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("gold_employment_treatment_mix")
)

StatementMeta(, 9088b372-c4ad-4da7-bd77-f8d720fc8c8a, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 90f86a17-431e-4063-907d-e5eb3260d5e2)

## 7. Validate KPI Outputs

Review the Gold KPI summary table to confirm that the pipeline outputs match the dashboard-level metrics.

In [12]:
gold_kpi_summary = spark.table("gold_kpi_summary")

display(gold_kpi_summary)

expected_metrics = [
    "Total Admissions",
    "High-Risk Admissions",
    "High Risk Rate",
    "Treatment Mismatch Rate",
    "High-Risk Alignment Rate",
    "High-Risk Delayed Admission Rate"
]

StatementMeta(, a94ada23-d6fc-470a-95cb-d80c1adf2e17, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4afefe6d-b32a-4d10-bbd9-65fc5cce934f)

SynapseWidget(Synapse.DataFrame, 7a751956-7586-4eb1-af4b-41d077ab93a1)

In [2]:
gold_employment_treatment_mix = spark.table("gold_employment_treatment_mix")


StatementMeta(, 9088b372-c4ad-4da7-bd77-f8d720fc8c8a, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5181456c-8f73-4d1d-8615-6b57661fd79a)